Libraries and Google drive

In [ ]:

!pip install -q -U bitsandbytes accelerate transformers neo4j sentence-transformers pydantic

import nltk
import os

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

print("✅ Setup Complete. Ready to load the model.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Path to NFR keywords YAML file
KEYWORDS_PATH = "/content/nfr_keywords - Copy.yaml"
# Path to fine-tuned NoRBERT folder
NOBERT_PATH = "/content/drive/MyDrive/NFR_Classifier"

# Check if paths exist
print(f"Checking paths...")
if os.path.exists(KEYWORDS_PATH): print(f"✅ Keywords found at {KEYWORDS_PATH}")
else: print(f"⚠️ Keywords file not found. System will use default rules.")

if os.path.exists(NOBERT_PATH): print(f"✅ NoRBERT model found at {NOBERT_PATH}")
else: print(f"⚠️ NoRBERT model not found. Stage 2 will rely on LLM/Rules.")

llm

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
import re

class LocalLLMHandler:
    def __init__(self):
        print("⏳ Loading Local LLM (Qwen2.5-7B-Instruct)...")
        model_id = "Qwen/Qwen2.5-7B-Instruct"

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print("✅ Local LLM Loaded.")

    def generate(self, prompt, temperature=0.2):

        safe_temp = max(temperature, 0.01)

        messages =[
            {"role": "system", "content": "You are a Requirements Engineering assistant. Output valid JSON only."},
            {"role": "user", "content": prompt}
        ]
        text_input = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = self.tokenizer([text_input], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                model_inputs.input_ids,
                attention_mask=model_inputs.attention_mask,
                pad_token_id=self.tokenizer.eos_token_id,
                max_new_tokens=512,
                temperature=safe_temp,
                do_sample=True
            )

        response = self.tokenizer.batch_decode(generated_ids[:, model_inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
        return self._extract_json(response)

    def _extract_json(self, text):
        try:
            clean_text = re.sub(r"```json|```", "", text).strip()
            match = re.search(r"\{.*\}", clean_text, re.DOTALL)
            if match: return json.loads(match.group(0))
        except: return None

# initialize the LLM
llm = LocalLLMHandler()

Stage 01

In [ ]:
import json
import re
import time
from nltk.tokenize import sent_tokenize
from pydantic import BaseModel, Field, ValidationError
from typing import List

# Data Model
class StructuredRequirement(BaseModel):
    actors: List[str] = Field(default_factory=list)
    action: str = ""
    object: str = ""
    constraints: List[str] = Field(default_factory=list)
    missing_info: List[str] = Field(default_factory=list)

class Stage1Structuring:
    def __init__(self, llm_handler):
        self.llm = llm_handler
        self.clean_pattern = re.compile(r'[^a-zA-Z0-9\s\.,;\'\"]')
        self.vague_regex = re.compile(r'\b(user-friendly|fast|efficient|robust|optimal|easy|appropriate|better|clean|soon)\b', re.IGNORECASE)

    def _preprocess(self, text):
        text = text.lower()
        text = self.clean_pattern.sub('', text)
        sentences = sent_tokenize(text)
        return " ".join(sentences)

    def _construct_prompt(self, text):
        #  PROMPT:
        return f"""You are a Requirements Extraction Tool. Extract fields from the text.

        OUTPUT JSON:
        {{
            "actors": ["List of actors"],
            "action": "Main action verb",
            "object": "Target object",
            "constraints": ["List of constraints"],
            "missing_info": ["List CRITICAL gaps only"]
        }}

        RULES:
        1. If the sentence has an Actor, Action, and Object, it is valid. Leave "missing_info" EMPTY.
        2. Only populate "missing_info" if the sentence is VAGUE (e.g. "make it fast") or missing a Subject.
        3. Do NOT ask for extra details like "security" or "frequency" if they are not mentioned.

        EXAMPLES:
        Input: "The System shall maintain a Patient Database."
        Output: {{ "actors": ["System"], "action": "maintain", "object": "Patient Database", "missing_info": [] }}

        Input: "System needs backup."
        Output: {{ "actors": ["System"], "action": "backup", "missing_info": ["What data?", "Frequency?"] }}

        TARGET: "{text}"
        """

    def _call_llm_with_retry(self, prompt):
        max_attempts = 3
        for attempt in range(max_attempts):
            try:
                raw_json = self.llm.generate(prompt, temperature=0.1)
                if not raw_json: raise ValueError("Empty response")
                return StructuredRequirement(**raw_json)
            except Exception as e:
                time.sleep(1)
        return StructuredRequirement(missing_info=["Analysis Failed"])

    def _calculate_completeness(self, data: StructuredRequirement):

        # 1. Field Presence (Weight: 0.4)
        fields = [data.actors, data.action, data.object, data.constraints]
        present = sum(1 for f in fields if f)
        score_presence = present / 4.0

        # 2. Vague Term Penalty (Weight: 0.3)
        vague_hits = sum(1 for c in data.constraints if self.vague_regex.search(c))
        # Penalty is capped at 1.0 (if 2 or more vague terms found)
        penalty_vague = min(vague_hits * 0.5, 1.0)

        # 3. Missing Info Penalty (Weight: 0.3)
        # Penalty scales with number of missing items detected
        penalty_missing = min(len(data.missing_info) * 0.5, 1.0)

        # Final Calculation
        score = (0.4 * score_presence) + (0.3 * (1.0 - penalty_vague)) + (0.3 * (1.0 - penalty_missing))
        return round(max(score, 0.0), 2)
    def analyze(self, raw_text):
        clean_text = self._preprocess(raw_text)
        structured = self._call_llm_with_retry(self._construct_prompt(clean_text))
        score = self._calculate_completeness(structured)

        questions = []
        if score < 0.80:
            q_prompt = f"Requirement: '{clean_text}'. Gaps: {structured.missing_info}. Generate 2 clarification questions in JSON: {{'questions': ['Q1', 'Q2']}}"
            q_res = self.llm.generate(q_prompt)
            questions = q_res.get('questions', []) if q_res else ["Please clarify."]

        return {
            "text": clean_text,
            "structured": structured.model_dump(),
            "score": score,
            "questions": questions
        }

Stage 02

In [ ]:
import os
import json
import yaml
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from collections import Counter


class Stage2Classification:
    def __init__(self, llm_handler, keywords_path, nobert_path):
        self.llm    = llm_handler
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        # ── 1. Load Rules ─────────────────────────────────────────
        self.rules = {}
        if os.path.exists(keywords_path):
            with open(keywords_path, 'r') as f:
                self.rules = yaml.safe_load(f)
        else:
            self.rules = {
                "Performance":     ["fast", "ms", "seconds", "response time",
                                    "throughput", "latency", "concurrent"],
                "Security":        ["encrypt", "password", "auth", "ssl",
                                    "tls", "access control", "certificate"],
                "Usability":       ["user-friendly", "easy to use", "intuitive",
                                    "accessible", "navigate", "interface"],
                "Availability":    ["available", "uptime", "fault tolerant",
                                    "failover", "downtime", "backup"],
                "Scalability":     ["scale", "grow", "expand",
                                    "concurrent users", "future"],
                "Maintainability": ["modular", "documented", "maintainable",
                                    "configurable", "extensible"],
                "Portability":     ["platform", "browser", "operating system",
                                    "compatible", "mobile", "desktop"],
            }

        # ── 2. Load  trained BERT model (NoRBERT slot) ────────
        self.nobert_ready = False
        self._load_nobert(nobert_path)

    # ══════════════════════════════════════════════════════════════
    # PRIVATE — model loader
    # ══════════════════════════════════════════════════════════════

    def _load_nobert(self, nobert_path):

        if not os.path.exists(nobert_path):
            print(f"[Stage2] NoRBERT path not found: {nobert_path}")
            return

        try:
            # Label map
            id2name_path = os.path.join(nobert_path, "id2name.json")
            if os.path.exists(id2name_path):
                with open(id2name_path) as f:
                    self.id2name = {int(k): v for k, v in json.load(f).items()}
            else:
                # Fallback — matches your training script's remap
                self.id2name = {
                    0:"FR", 1:"Availability",  2:"Fault Tolerance", 3:"Legal",  4:"Look & Feel",
                    5:"Maintainability", 6:"Operational",  7:"Performance", 8:"Portability",  9:"Scalability",
                    10:"Security", 11:"Usability"
                }

            # Tuned thresholds
            thresh_path = os.path.join(nobert_path, "thresholds.json")
            if os.path.exists(thresh_path):
                with open(thresh_path) as f:
                    self.thresholds = {int(k): v for k, v in json.load(f).items()}
            else:
                self.thresholds = {i: 0.5 for i in range(len(self.id2name))}

            # Tokenizer + model
            self.tokenizer = AutoTokenizer.from_pretrained(nobert_path)
            self.nobert    = AutoModelForSequenceClassification.from_pretrained(
                nobert_path
            ).to(self.device)
            self.nobert.eval()
            self.nobert_ready = True

            print(f"[Stage2] ✓ NoRBERT loaded ({len(self.id2name)} classes)")
            print(f"[Stage2]   Labels     : {self.id2name}")
            print(f"[Stage2]   Thresholds : {self.thresholds}")

        except Exception as e:
            print(f"[Stage2] ✗ NoRBERT load failed: {e}")
            self.nobert_ready = False

    # ══════════════════════════════════════════════════════════════
    # CLASSIFIER 1 — Rule-based
    # Returns: (fr_nfr: str, subtype: str)
    # ══════════════════════════════════════════════════════════════

    def classify_rules(self, text):
        text_lower = text.lower()
        scores     = {cat: 0 for cat in self.rules}

        for cat, keywords in self.rules.items():
            if any(kw in text_lower for kw in keywords):
                scores[cat] += 1

        if sum(scores.values()) == 0:
            return "FR", "None"

        best_subtype = max(scores, key=scores.get)
        return "NFR", best_subtype

    # ══════════════════════════════════════════════════════════════
    # CLASSIFIER 2 — LLM
    # Returns: (fr_nfr: str, subtype: str)
    # ══════════════════════════════════════════════════════════════

    def classify_llm(self, text):
        prompt = f"""You are an expert Requirements Engineer specializing in
classifying software requirements.

DEFINITIONS:
- FR (Functional Requirement): Describes WHAT the system does.
  Specifies a behavior, feature, or function. Uses action verbs
  like "shall allow", "shall process", "shall display", "shall store".

- NFR (Non-Functional Requirement): Describes HOW WELL the system
  performs. Specifies quality attributes, constraints, or standards.

NFR CATEGORIES AND STRONG SIGNALS:
- Performance: response time, speed, latency, throughput, seconds,
  milliseconds, concurrent users, load, capacity
- Security: encrypt, authenticate, authorize, password, access control,
  SSL, TLS, certificate, audit, log
- Usability: user-friendly, easy to use, intuitive, accessible,
  learn, navigate, interface appearance
- Reliability: available, uptime, fault tolerant, recover, backup,
  failover, MTBF, downtime
- Maintainability: modular, documented, testable, configurable,
  upgradeable, extensible
- Portability: platform, browser, operating system, device,
  compatible, mobile, desktop
- Scalability: scale, grow, expand, additional users, future

CRITICAL RULE: If the requirement mentions TIME LIMITS, PERCENTAGES,
QUANTITIES as constraints, SIZE LIMITS, or QUALITY STANDARDS —
it is almost certainly NFR even if it uses "shall".

EXAMPLES:
"The system shall refresh the display every 60 seconds." → NFR (Performance)
"The system shall encrypt all passwords using SHA-256."  → NFR (Security)
"The system shall allow users to log in."                → FR
"The application shall be available 99.9% of the time." → NFR (Reliability)
"The system shall process 1000 transactions per second." → NFR (Performance)
"The system shall display a list of products."           → FR

NOW CLASSIFY THIS REQUIREMENT:
"{text}"

Respond in JSON only, no explanation outside JSON:
{{
  "type": "FR" or "NFR",
  "subtype": null if FR, or one of [Performance, Security, Usability,
             Reliability, Maintainability, Portability, Scalability] if NFR,
  "confidence": "high", "medium", or "low",
  "reasoning": "one sentence explanation"
}}"""

        data = self.llm.generate(prompt)
        if data:
            return data.get('type', "Unknown"), data.get('subtype') or "None"
        return "Unknown", "None"

    # ══════════════════════════════════════════════════════════════
    # CLASSIFIER 3 — Trained BERT (NoRBERT slot)
    # Returns: (fr_nfr: str, subtype: str)
    # ══════════════════════════════════════════════════════════════

    def classify_nobert(self, text):
        if not self.nobert_ready:
            return "Unknown", "None"

        try:
            inputs = self.tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128,
            ).to(self.device)

            with torch.no_grad():
                logits = self.nobert(**inputs).logits
                probs  = torch.softmax(logits, dim=1).squeeze().cpu().numpy()

            # Apply tuned thresholds (from your training script section 14)
            pred_id = int(np.argmax(probs))
            for cid, thresh in self.thresholds.items():
                if thresh < 0.5 and probs[cid] >= thresh:
                    pred_id = cid
                    break

            label = self.id2name.get(pred_id, "Unknown")

            if label == "FR":
                return "FR", "None"
            else:
                return "NFR", label     # e.g. "NFR", "PE"

        except Exception as e:
            print(f"[Stage2] NoRBERT inference error: {e}")
            return "Unknown", "None"

    # ══════════════════════════════════════════════════════════════
    # SUBTYPE RESOLVER
    # Priority: BERT → LLM → Rules
    # ══════════════════════════════════════════════════════════════

    def _resolve_subtype(self, r1, r2, r3):

        for result in [r3, r2, r1]:
            subtype = result[1]
            if subtype and subtype not in ("None", "NFR_Subtype", "Unknown"):
                return subtype
        return "None"

    # ══════════════════════════════════════════════════════════════
    # MAIN ENTRY POINT — output structure unchanged
    # ══════════════════════════════════════════════════════════════

    def process(self, req_data):
        text = req_data['text']

        # Run all three classifiers
        r1 = self.classify_rules(text)
        r2 = self.classify_llm(text)
        r3 = self.classify_nobert(text)

        # ── FR / NFR voting (identical to original) ───────────────
        votes       = [r1[0], r2[0], r3[0]]
        clean_votes = [v for v in votes if v != "Unknown"]

        if not clean_votes:
            winner, conf = "Unknown", "LOW"
        else:
            winner, count = Counter(clean_votes).most_common(1)[0]
            conf = "HIGH" if count == 3 else "MEDIUM" if count == 2 else "LOW"

        # ── Subtype (only if NFR) ─────────────────────────────────
        subtype = "None"
        if winner == "NFR":
            subtype = self._resolve_subtype(r1, r2, r3)

        # ── Return — same structure as original ───────────────────
        return {
            "id":   req_data['id'],
            "text": text,
            "classification": {
                "final_label": winner,
                "subtype":     subtype,
                "confidence":  conf,
                "votes": {
                    "Rules":   r1[0],
                    "LLM":     r2[0],
                    "NoRBERT": r3[0],
                }
            }
        }

Stage 03

In [ ]:
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import numpy as np
import nltk

embedder = SentenceTransformer('all-MiniLM-L6-v2')

class Stage3GraphBuilder:
    def __init__(self, uri, user, pwd, llm_handler):
        self.driver = GraphDatabase.driver(uri, auth=(user, pwd))
        self.llm = llm_handler

        # Universal structural dependency words
        self.structural_deps = ["requires", "depends on", "after", "before", "prerequisite", "based on"]

    def _extract_noun_phrases(self, text):

        import re
        tokens = nltk.word_tokenize(text.lower())
        tags = nltk.pos_tag(tokens)

        phrases = []
        current_phrase = []

        for word, tag in tags:
            if tag.startswith('NN') or tag.startswith('JJ'):
                current_phrase.append(word)
            else:
                if len(current_phrase) >= 2:
                    phrases.append(' '.join(current_phrase))
                elif len(current_phrase) == 1:
                    phrases.append(current_phrase[0])
                current_phrase = []

        if current_phrase:
            if len(current_phrase) >= 2:
                phrases.append(' '.join(current_phrase))
            else:
                phrases.append(current_phrase[0])

        ignore = {'system', 'user', 'application', 'software',
                  'shall', 'must', 'should', 'will', 'following',
                  'new', 'current', 'existing', 'available', 'able'}
        return {p for p in phrases if p not in ignore and len(p) > 2}

    def _detect_numerical_conflict(self, text_a, text_b):

        import re

        # Domain concept groups - if both reqs share a concept
        concept_groups = [
            ['session', 'login', 'authentication',
             'inactivity', 'timeout', 'persistent', 'expire'],
            ['response', 'latency', 'processing',
             'load', 'performance', 'throughput'],
            ['password', 'attempt', 'lockout', 'retry'],
            ['encrypt', 'tls', 'ssl', 'cipher', 'key'],
            ['memory', 'storage', 'disk', 'size', 'capacity'],
            ['user', 'concurrent', 'connection', 'request'],
        ]

        text_a_lower = text_a.lower()
        text_b_lower = text_b.lower()

        # Extract all time/quantity values
        pattern = r'(\d+(?:\.\d+)?)\s*(millisecond|second|minute|hour|day|week|month|year|ms|kb|mb|gb|%|percent)s?'

        values_a = re.findall(pattern, text_a_lower)
        values_b = re.findall(pattern, text_b_lower)

        if not values_a or not values_b:
            return False, None

        # Check if both reqs share a concept group
        for group in concept_groups:
            a_has = any(k in text_a_lower for k in group)
            b_has = any(k in text_b_lower for k in group)

            if a_has and b_has:
                # Same concept, different numeric constraints = conflict
                units_a = {unit for _, unit in values_a}
                units_b = {unit for _, unit in values_b}
                shared_units = units_a.intersection(units_b)

                if shared_units:
                    nums_a = {float(n) for n, u in values_a
                              if u in shared_units}
                    nums_b = {float(n) for n, u in values_b
                              if u in shared_units}
                    if nums_a != nums_b:
                        return True, (f"Numerical conflict on shared concept "
                                     f"'{group[0]}': "
                                     f"{values_a} vs {values_b}")
                else:
                    # Different units but same concept - still a conflict
                    if values_a != values_b:
                        return True, (f"Constraint conflict on shared concept "
                                     f"'{group[0]}': "
                                     f"{values_a} vs {values_b}")

        return False, None

    def _detect_negation_conflict(self, text_a, text_b):
        import re

        # Extract the core action from each requirement
        shall_pattern = r'shall\s+(not\s+)?(\w+(?:\s+\w+){0,3})'
        must_pattern  = r'must\s+(not\s+)?(\w+(?:\s+\w+){0,3})'

        def extract_actions(text):
            actions = []
            for pattern in [shall_pattern, must_pattern]:
                matches = re.findall(pattern, text.lower())
                for negated, action in matches:
                    actions.append({
                        'action': action.strip(),
                        'negated': bool(negated)
                    })
            return actions

        actions_a = extract_actions(text_a)
        actions_b = extract_actions(text_b)

        for act_a in actions_a:
            for act_b in actions_b:
                # Check if same action but one negated and one not
                words_a = set(act_a['action'].split())
                words_b = set(act_b['action'].split())
                overlap = words_a.intersection(words_b)

                if len(overlap) >= 2 and act_a['negated'] != act_b['negated']:
                    return True, (f"Negation conflict: "
                                 f"'{act_a['action']}' vs "
                                 f"'not {act_b['action']}'")

        return False, None

    def create_nodes(self, reqs):
        texts = [r['text'] for r in reqs]
        embeddings = embedder.encode(texts)
        with self.driver.session() as session:
            session.run("MATCH (n) DETACH DELETE n")
            for i, r in enumerate(reqs):
                c = r['classification']
                session.run("""
                CREATE (r:Requirement {id: $id, text: $text, type: $type, subtype: $sub, conf: $conf})
                """, id=r['id'], text=r['text'], type=c['final_label'], sub=c['subtype'], conf=c['confidence'])
        return reqs, embeddings

    def build_relationships(self, reqs, embeddings):

        dep_links = []
        conflict_links = []
        n = len(reqs)

        with self.driver.session() as session:
            for i in range(n):
                for j in range(i+1, n):  # i+1 avoids duplicates
                    req_a = reqs[i]
                    req_b = reqs[j]
                    text_a = req_a['text']
                    text_b = req_b['text']

                    # ── SEMANTIC SIMILARITY ──────────────────────────
                    sim = np.dot(embeddings[i], embeddings[j]) / (
                        np.linalg.norm(embeddings[i]) *
                        np.linalg.norm(embeddings[j]))

                    # Only analyze pairs with some semantic relationship
                    # Low threshold 0.30 - LLM decides, not math
                    if sim < 0.30:
                        continue

                    # ── LLM PRIMARY ANALYSIS ─────────────────────────
                    # Ask LLM to analyze dependency AND conflict
                    # in a single call to save API time
                    analysis_prompt = f"""Analyze these two software
requirements for logical relationships.

Requirement A (id={req_a['id']}): "{text_a}"
Requirement B (id={req_b['id']}): "{text_b}"

Answer these questions:

1. DEPENDENCY: Does A logically require B to exist first?
   (A constrains B, A measures quality of B,
    A uses functionality provided by B)

2. REVERSE DEPENDENCY: Does B logically require A to exist first?

3. CONFLICT: Do A and B contradict each other?
   (They cannot both be satisfied simultaneously,
    they specify different values for the same constraint,
    one says shall and the other says shall not
    for the same behavior)

Reply in JSON only:
{{
  "a_depends_on_b": true or false,
  "b_depends_on_a": true or false,
  "conflict": true or false,
  "conflict_severity": "high", "medium", or "low",
  "reasoning": "one sentence explaining any relationship found"
}}"""

                    llm_result = self.llm.generate(
                        analysis_prompt, temperature=0.0)

                    if not llm_result:
                        continue

                    # ── DEPENDENCY A -> B ─────────────────────────────
                    if llm_result.get('a_depends_on_b') is True:
                        dep_votes = ['LLM_Primary']

                        # Semantic as validator
                        if sim > 0.50:
                            dep_votes.append(f'Semantic({sim:.2f})')

                        # Structural keyword as validator
                        if any(k in text_a.lower()
                               for k in self.structural_deps):
                            dep_votes.append('Structural_Keyword')

                        # Accept if LLM confirmed
                        # (LLM alone is sufficient as primary detector)
                        session.run(
                            """MATCH (a:Requirement {id:$src}),
                                   (b:Requirement {id:$tgt})
                               MERGE (a)-[r:DEPENDS_ON]->(b)
                               SET r.methods=$v, r.sim=$s,
                                   r.reason=$reason""",
                            src=req_a['id'],
                            tgt=req_b['id'],
                            v=dep_votes,
                            s=float(sim),
                            reason=llm_result.get('reasoning', ''))
                        dep_links.append((req_a['id'], req_b['id']))
                        print(f"   dep {req_a['id']} -> {req_b['id']} "
                              f"| {dep_votes} "
                              f"| {llm_result.get('reasoning','')[:60]}")

                    # ── DEPENDENCY B -> A ─────────────────────────────
                    if llm_result.get('b_depends_on_a') is True:
                        dep_votes = ['LLM_Primary']

                        if sim > 0.50:
                            dep_votes.append(f'Semantic({sim:.2f})')

                        if any(k in text_b.lower()
                               for k in self.structural_deps):
                            dep_votes.append('Structural_Keyword')

                        session.run(
                            """MATCH (a:Requirement {id:$src}),
                                   (b:Requirement {id:$tgt})
                               MERGE (a)-[r:DEPENDS_ON]->(b)
                               SET r.methods=$v, r.sim=$s,
                                   r.reason=$reason""",
                            src=req_b['id'],
                            tgt=req_a['id'],
                            v=dep_votes,
                            s=float(sim),
                            reason=llm_result.get('reasoning', ''))
                        dep_links.append((req_b['id'], req_a['id']))
                        print(f"   dep {req_b['id']} -> {req_a['id']} "
                              f"| {dep_votes} "
                              f"| {llm_result.get('reasoning','')[:60]}")

                    # ── CONFLICT DETECTION ────────────────────────────
                    if llm_result.get('conflict') is True:

                        # Rule-based validators for conflict
                        conflict_votes = ['LLM_Primary']

                        # Numerical contradiction validator
                        import re
                        nums_a = re.findall(
                            r'\d+(?:\.\d+)?\s*'
                            r'(?:second|minute|hour|day|ms|%|mb|gb)s?',
                            text_a.lower())
                        nums_b = re.findall(
                            r'\d+(?:\.\d+)?\s*'
                            r'(?:second|minute|hour|day|ms|%|mb|gb)s?',
                            text_b.lower())

                        if nums_a and nums_b and nums_a != nums_b:
                            conflict_votes.append(
                                f'Numerical({nums_a[0]} vs {nums_b[0]})')

                        # Negation validator
                        has_negation = (
                            ('shall not' in text_a.lower() and
                             'shall' in text_b.lower() and
                             'shall not' not in text_b.lower()) or
                            ('shall not' in text_b.lower() and
                             'shall' in text_a.lower() and
                             'shall not' not in text_a.lower())
                        )
                        if has_negation:
                            conflict_votes.append('Negation_Pattern')

                        severity = llm_result.get(
                            'conflict_severity', 'medium')

                        session.run(
                            """MATCH (a:Requirement {id:$src}),
                                   (b:Requirement {id:$tgt})
                               MERGE (a)-[r:CONFLICTS_WITH]->(b)
                               SET r.methods=$v, r.severity=$sev,
                                   r.reason=$reason""",
                            src=req_a['id'],
                            tgt=req_b['id'],
                            v=conflict_votes,
                            sev=severity,
                            reason=llm_result.get('reasoning', ''))
                        conflict_links.append(
                            (req_a['id'], req_b['id']))
                        print(f"   conflict {req_a['id']} <-> "
                              f"{req_b['id']} [{severity}] "
                              f"| {conflict_votes} "
                              f"| {llm_result.get('reasoning','')[:60]}")

        print(f"\n   Total dependencies: {len(dep_links)}")
        print(f"   Total conflicts: {len(conflict_links)}")
        return dep_links

Matric calculator

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

class MetricsCalc:
    def __init__(self):
        self.s1_truth, self.s1_pred = [], []
        self.s2_truth, self.s2_pred = [], []
        self.s3_truth, self.s3_pred = set(), set()
        self.start_time = 0; self.end_time = 0
        self.auto_count = 0; self.total_count = 0

    def add_s1(self, truth_complete, score):
        self.s1_truth.append(1 if truth_complete else 0)
        self.s1_pred.append(1 if score >= 0.8 else 0)

    def add_s2(self, truth_type, pred_type):
        self.s2_truth.append(truth_type)
        self.s2_pred.append(pred_type)

    def set_s3(self, truth_links, pred_links):
        self.s3_truth = set(truth_links)
        self.s3_pred = set(pred_links)

    def print_results(self, name):
        duration = self.end_time - self.start_time
        print(f"\n📊 --- FINAL METRICS REPORT: {name} ---")
        print("="*50)

        # Stage 1
        s1_acc = accuracy_score(self.s1_truth, self.s1_pred)
        print(f"1️⃣ Stage 1 (Completeness)")
        print(f"   • Accuracy: {s1_acc:.2%}")

        # Stage 2
        s2_acc = accuracy_score(self.s2_truth, self.s2_pred)
        print(f"2️⃣ Stage 2 (Classification)")
        print(f"   • Voting Accuracy: {s2_acc:.2%}")

        # Stage 3
        tp = len(self.s3_pred.intersection(self.s3_truth))
        fp = len(self.s3_pred - self.s3_truth)
        fn = len(self.s3_truth - self.s3_pred)

        prec = tp/(tp+fp) if (tp+fp)>0 else 0
        rec = tp/(tp+fn) if (tp+fn)>0 else 0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0

        print(f"3️⃣ Stage 3 (Traceability)")
        print(f"   • Precision: {prec:.2%}")
        print(f"   • Recall:    {rec:.2%}")
        print(f"   • F1-Score:  {f1:.2f}")

        # System
        automation = self.auto_count / self.total_count if self.total_count > 0 else 0
        print(f"⚙️ System Performance")
        print(f"   • Processing Time: {duration:.2f}s")
        print(f"   • Automation Rate: {automation:.1%}")
        print("="*50)

In [ ]:
test_batch = [
     # ROOT: login
    {"id": "BANK_01",
     "text": "The system shall allow registered users to log in using their email address and password.",
     "truth": {"complete": True, "type": "FR"},
     "links": []},

    # DEPENDS ON: BANK_01 (needs login first)
    {"id": "BANK_02",
     "text": "The system shall display the account dashboard showing current balance, recent transactions, and quick transfer options after successful login.",
     "truth": {"complete": True, "type": "FR"},
     "links": ["BANK_01"]},

    # DEPENDS ON: BANK_01 (lockout is part of login)
    {"id": "BANK_03",
     "text": "The system shall lock the user account after 5 consecutive failed login attempts and send an unlock email to the registered address.",
     "truth": {"complete": True, "type": "FR"},
     "links": ["BANK_01"]},

    # DEPENDS ON: BANK_02 (transfer visible on dashboard)
    {"id": "BANK_04",
     "text": "The system shall allow authenticated users to transfer funds between their own accounts by entering an amount and selecting source and destination accounts.",
     "truth": {"complete": True, "type": "FR"},
     "links": ["BANK_02"]},

    # DEPENDS ON: BANK_04 (SMS sent after transfer)
    {"id": "BANK_05",
     "text": "The system shall send an SMS confirmation to the user registered mobile number containing the transfer amount, destination account last 4 digits, and timestamp after every successful fund transfer.",
     "truth": {"complete": True, "type": "FR"},
     "links": ["BANK_04"]},

    # NFR: session timeout - DEPENDS ON BANK_01
    {"id": "BANK_06",
     "text": "All user authentication sessions shall expire after 15 minutes of inactivity and require re-authentication before any further account access.",
     "truth": {"complete": True, "type": "NFR"},
     "links": ["BANK_01"]},

    # NFR: performance - DEPENDS ON BANK_04
    {"id": "BANK_07",
     "text": "Fund transfer transactions shall complete end-to-end processing within 3 seconds for 95% of requests under normal load of up to 10,000 concurrent users.",
     "truth": {"complete": True, "type": "NFR"},
     "links": ["BANK_04"]},

    # NFR: security - DEPENDS ON BANK_01 + BANK_04
    {"id": "BANK_08",
     "text": "The system shall encrypt all data transmitted between the client browser and server using TLS 1.3 or higher including login credentials and transaction data.",
     "truth": {"complete": True, "type": "NFR"},
     "links": ["BANK_01", "BANK_04"]},

    # NFR: CONFLICTS WITH BANK_06 (30 day vs 15 min session)
    {"id": "BANK_09",
     "text": "The system shall maintain persistent login sessions for 30 days on trusted devices to improve user convenience without requiring repeated authentication.",
     "truth": {"complete": True, "type": "NFR"},
     "links": ["BANK_01"]},

    # VAGUE: Stage 1 trap
    {"id": "BANK_10",
     "text": "The system must be secure and easy to use for all customers.",
     "truth": {"complete": False, "type": "NFR"},
     "links": ["BANK_01"]},

    # VAGUE: Stage 1 trap
    {"id": "BANK_11",
     "text": "The dashboard should load fast.",
     "truth": {"complete": False, "type": "NFR"},
     "links": ["BANK_02"]},

    # DEPENDS ON: BANK_02 (PDF from dashboard)
    {"id": "BANK_12",
     "text": "The system shall generate a downloadable PDF statement for any selected date range showing all transactions, opening balance, closing balance, and account holder details.",
     "truth": {"complete": True, "type": "FR"},
     "links": ["BANK_02"]},
]

In [ ]:
from google.colab import userdata
import time

calc = MetricsCalc()
stage1 = Stage1Structuring(llm)
stage2 = Stage2Classification(llm, KEYWORDS_PATH, NOBERT_PATH)

try:
    uri = userdata.get('NEO4J_URI')
    user = userdata.get('NEO4J_USERNAME')
    pwd = userdata.get('NEO4J_PASSWORD')
    stage3 = Stage3GraphBuilder(uri, user, pwd, llm)

    calc.start_time = time.time()
    calc.total_count = len(test_batch)

    # --- STAGE 1 ---
    print("\n🔹 STARTING STAGE 1...")
    s1_results = []
    for r in test_batch:
        print(f"\nProcessing {r['id']}...")
        res = stage1.analyze(r['text'])

        # Metric Log
        calc.s1_truth.append(1 if r['truth']['complete'] else 0)
        calc.s1_pred.append(1 if res['score'] >= 0.8 else 0)
        if res['score'] >= 0.8: calc.auto_count += 1

        current_text = r['text']
        iteration = 0
        while res['score'] < 0.80 and iteration < 3:
            print(f"⚠️ Incomplete (Score: {res['score']})")
            print(f"   Questions: {res['questions']}")
            user_input = input(">> Enter Clarification: ")
            # TYPE: "Back up the Patient Database daily"
            current_text += " " + user_input
            res = stage1.analyze(current_text)
            iteration += 1

        print(f"✅ Final: {current_text}")
        s1_results.append({"id": r['id'], "text": current_text})

    # --- STAGE 2 ---
    print("\n🔹 STARTING STAGE 2...")
    s2_results = []
    for i, r in enumerate(s1_results):
        res = stage2.process(r)
        truth = test_batch[i]['truth']['type']
        calc.s2_truth.append(truth)
        calc.s2_pred.append(res['classification']['final_label'])

        c = res['classification']
        print(f"   {r['id']} -> {c['final_label']} ({c['subtype']}) | Votes: {c['votes']}")
        s2_results.append(res)

    # --- STAGE 3 ---
    print("\n🔹 STARTING STAGE 3...")
    reqs, embs = stage3.create_nodes(s2_results)
    found_links = stage3.build_relationships(reqs, embs)

    for src, tgt in found_links:
        print(f"   🔗 {src} -> {tgt}")

    truth_links = []
    for r in test_batch:
        for t in r['links']: truth_links.append((r['id'], t))
    calc.set_s3(truth_links, found_links)

    calc.end_time = time.time()
    calc.print_results("TEST BATCH")
    stage3.driver.close()

except Exception as e:
    print(f"❌ Error: {e}")

Stage 2 Evaluation

In [ ]:
# ============================================================
# PROMISE NFR Dataset — Stage 2 Evaluation (v2)
# Compatible with the upgraded Stage2Classification class
# ============================================================

import pandas as pd
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

# ── 1. Load PROMISE dataset ──────────────────────────────────
PROMISE_PATH = "/content/Promise_NFR_dataset_orginal.csv"   # adjust filename if needed

df = pd.read_csv(PROMISE_PATH, sep=";")
df.columns = df.columns.str.strip()

print(f"✅ Loaded {len(df)} requirements")

# ── 2. Ground truth labels ───────────────────────────────────
def get_binary_label(row):
    return "FR" if row["F"] == 1 else "NFR"

# Map PROMISE class codes to readable subtype names
SUBTYPE_MAP = {
    "F":  "FR",
    "PE": "Performance",
    "SC": "Security",
    "SE": "Security",
    "US": "Usability",
    "A":  "Availability",
    "FT": "Reliability",
    "L":  "Legal",
    "LF": "Look_and_Feel",
    "MN": "Maintainability",
    "O":  "Operational",
    "PO": "Portability",
}

df["truth_binary"]  = df.apply(get_binary_label, axis=1)
df["truth_subtype"] = df["class"].str.strip().map(SUBTYPE_MAP).fillna("Unknown")

# ── 3. Held-out test split (last 125 = 20%) ──────────────────
test_df = df.iloc[-125:].reset_index(drop=True)
print(f"🧪 Test partition: {len(test_df)} requirements")
print(test_df["truth_binary"].value_counts())

# ── 4. Run upgraded Stage 2 on each requirement ──────────────
# FIX 1: stage2_eval must be created with the NEW class definition
#         Make sure you run the new Stage2Classification cell first!
stage2_eval = Stage2Classification(llm, KEYWORDS_PATH, NOBERT_PATH)

rules_preds    = []
llm_preds      = []
nobert_preds   = []
ensemble_preds = []

# For subtype evaluation (NFR requirements only)
subtype_truths = []
subtype_preds  = []

print("\n🔹 Running classifiers on 125 requirements...")

for i, row in test_df.iterrows():
    text = str(row["RequirementText"]).strip()

    # FIX 2: all three methods still return tuples (label, subtype)
    # — unchanged interface, so unpacking works the same way
    rules_label,  rules_sub  = stage2_eval.classify_rules(text)
    llm_label,    llm_sub    = stage2_eval.classify_llm(text)
    nobert_label, nobert_sub_raw = stage2_eval.classify_nobert(text)

# Map NoRBERT's raw codes to full names (same as truth labels)
    NOBERT_CODE_MAP = {
        "FR":"FR", "A":"Availability", "FT":"Reliability", "L":"Legal",
        "LF":"Look_and_Feel", "MN":"Maintainability", "O":"Operational",
        "PE":"Performance", "PO":"Portability", "SC":"Security",
        "SE":"Security", "US":"Usability"
    }
    nobert_sub = NOBERT_CODE_MAP.get(nobert_sub_raw, nobert_sub_raw)

    # Voting (same logic as process())
    votes       = [v for v in [rules_label, llm_label, nobert_label]
                   if v != "Unknown"]
    fr_count    = votes.count("FR")
    nfr_count   = votes.count("NFR")
    ensemble    = "FR" if fr_count >= nfr_count else "NFR"

    # Subtype resolution using new priority: NoRBERT > LLM > Rules
    # All three now use the same full-name format — comparison works correctly
    subtype = "None"
    if ensemble == "NFR":
        for sub in [nobert_sub, llm_sub, rules_sub]:
            if sub and sub not in ("None", "NFR_Subtype", "Unknown", "FR"):
                subtype = sub
                break

    # These appends should always happen for every requirement
    rules_preds.append(rules_label)
    llm_preds.append(llm_label)
    nobert_preds.append(nobert_label)
    ensemble_preds.append(ensemble)

    # Collect subtype data for NFR rows only (this remains conditional)
    if row["truth_binary"] == "NFR":
        subtype_truths.append(row["truth_subtype"])
        subtype_preds.append(subtype)

    if (i + 1) % 25 == 0:
        print(f"   {i+1}/125 done...")

print("✅ Done.")

# ── 5. Binary FR/NFR metrics ─────────────────────────────────
y_true = test_df["truth_binary"].tolist()

def print_metrics(y_true, y_pred, name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label="NFR", zero_division=0)
    rec  = recall_score(y_true, y_pred,    pos_label="NFR", zero_division=0)
    f1   = f1_score(y_true, y_pred,        pos_label="NFR", zero_division=0)
    print(f"  {name:<30} Acc:{acc:.1%}  Prec:{prec:.1%}  "
          f"Rec:{rec:.1%}  F1:{f1:.3f}")
    return dict(model=name, accuracy=acc, precision=prec, recall=rec, f1=f1)

print("\n" + "="*70)
print("📊  BINARY FR/NFR RESULTS  (PROMISE test partition, n=125)")
print("="*70)
rows = []
rows.append(print_metrics(y_true, rules_preds,    "Rules Classifier"))
rows.append(print_metrics(y_true, llm_preds,      "LLM (Qwen2.5-7B)"))
rows.append(print_metrics(y_true, nobert_preds,   "NoRBERT (fine-tuned)"))
rows.append(print_metrics(y_true, ensemble_preds, "Ensemble (Voting) ★"))

# ── 6. Ensemble confusion matrix ─────────────────────────────
print("\n📋 Ensemble Confusion Matrix:")
cm = confusion_matrix(y_true, ensemble_preds, labels=["FR","NFR"])
print(pd.DataFrame(cm,
      index=["True FR","True NFR"],
      columns=["Pred FR","Pred NFR"]).to_string())

# ── 7. NFR subtype accuracy (new — enabled by upgraded NoRBERT) ──
print("\n" + "="*70)
print("📊  NFR SUBTYPE ACCURACY  (NFR requirements only)")
print("="*70)
if subtype_truths:
    sub_acc = accuracy_score(subtype_truths, subtype_preds)
    print(f"  Subtype Accuracy: {sub_acc:.1%}  ({len(subtype_truths)} NFR requirements)")
    print("\n  Per-subtype breakdown:")
    print(classification_report(subtype_truths, subtype_preds, zero_division=0))
else:
    print("  No NFR subtype data collected.")

# ── 8. Summary table (Table 3 for thesis) ────────────────────
print("\n" + "="*70)
print("TABLE 3 — Classification Performance (PROMISE NFR, n=125)")
print("="*70)
print(f"{'Model':<32} {'Accuracy':>9} {'Precision':>10} "
      f"{'Recall':>8} {'F1':>8}")
print("-"*70)
for r in rows:
    star = " ★" if "Ensemble" in r["model"] else ""
    print(f"{r['model']+star:<32} {r['accuracy']:>9.1%} "
          f"{r['precision']:>10.1%} {r['recall']:>8.1%} {r['f1']:>8.3f}")
print("="*70)

# ── 9. Error analysis ─────────────────────────────────────────
test_df["ensemble_pred"] = ensemble_preds
errors = test_df[test_df["truth_binary"] != test_df["ensemble_pred"]]
print(f"\n🔍 Ensemble errors: {len(errors)}/125")
print(f"   FP (pred NFR, true FR): {len(errors[errors['ensemble_pred']=='NFR'])}")
print(f"   FN (pred FR, true NFR): {len(errors[errors['ensemble_pred']=='FR'])}")
if len(errors) > 0:
    print("\n   Misclassified requirements (first 5):")
    for _, e in errors.head(5).iterrows():
        print(f"   [True:{e['truth_binary']} | Pred:{e['ensemble_pred']}] "
              f"[Class:{e['class']}] {str(e['RequirementText'])[:70]}...")